In [2]:
import kagglehub
import os

In [3]:
path = kagglehub.dataset_download("shanti33/womens-health-pubmed-dataset-1786-2026")

100%|██████████| 1.13G/1.13G [00:16<00:00, 74.8MB/s]

Extracting files...


In [4]:
files = os.listdir(path)
file_path = os.path.join(path, files[0])
print(path)
print(files)

/root/.cache/kagglehub/datasets/shanti33/womens-health-pubmed-dataset-1786-2026/versions/1
['key_pubmed.csv']


In [37]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, regexp_replace, translate
from pyspark.sql.window import Window

In [6]:
spark = SparkSession.builder.master("local[*]").appName("WH Arciles Cleaner").getOrCreate()

In [7]:
df = spark.read.csv(file_path, header=True, inferSchema=True)

In [8]:
# Get some basic info
print(df.columns)
print(df.printSchema())
print(df.select("doi").count())
print(df.select("doi").distinct().count())

['title', 'doi', 'abstract', 'journal', 'date', 'authors', 'year', 'keyword']
root
 |-- title: string (nullable = true)
 |-- doi: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- journal: string (nullable = true)
 |-- date: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- year: string (nullable = true)
 |-- keyword: string (nullable = true)

None
4842236
3477406


In [ ]:
df.head(40)

[Row(title='IV. Prolapsus of the Rectum in Children.', doi='10.1097/00000658-190001000-00028', abstract=None, journal='Annals of surgery', date='1900-03-01', authors="['C GCumston']", year='1900.0', keyword="['hemorrhoids']"), Row(title='HEMIANOPSIA IN MIGRAINE.', doi='10.1126/science.11.279.713-a', abstract=None, journal='Science (New York, N.Y.)', date='1900-05-04', authors="['J MBaldwin']", year='1900.0', keyword="['migraine']"), Row(title='An attempt to estimate fatigue of the cerebral cortex when caused by electrical excitation.', doi='10.1113/jphysiol.1901.sp000832', abstract=None, journal='The Journal of physiology', date='1901-02-28', authors="['A GLevy']", year='1901.0', keyword="['fatigue']"), Row(title='VII. Report of a Case of Recovery after Abdominal Section for Multiple Gunshot Wounds of the Abdomen: Perforations of Liver, Gall-Bladder, Hepatic Flexure of Colon, Two of the Small Intestines, One of Rectum (Deep in Pelvis), with Active and Excessive Haemorrhage.', doi='10.1

In [ ]:
df.tail(15)

[Row(title='Bridging environmental pollution and immune dysregulation: How microplastics disrupt gut homeostasis to promote allergic diseases.', doi='10.1080/21505594.2026.2690759', abstract='"Parallel increases in microplastic (MP) pollution and allergic diseases suggest an environmental link in allergy pathogenesis. While intestinal barrier dysfunction and dysbiosis are hallmark features of allergic diseases, the mechanisms by which persistent MPs drive these processes remain unclear. This review proposes a mechanistic pathway where orally ingested MPs disrupt the intestinal barrier through physical and chemical toxicity, precipitating functional dysbiosis. Specifically, MPs suppress key short-chain fatty acid (SCFA)-producing commensals. The resulting SCFA deficit impairs the ""gut-airway axis', journal='"" weakening respiratory immune tolerance and increasing allergen susceptibility. We further argue for integrating gut health assessments into standardized allergy diagnostics and t

In [ ]:
# Looks like some records don't have doi but rest columns looks fine, -> df_null_doi
# Some records are malformed but they contains doi in one of other columns -> df_malformed
# Some records with correct doi have shufled values in others columns -> df_date_malformed
# Records with fine doi and date format -> df_fine

In [21]:
regex_doi = r"^\d{2}\.\d+/"
regex_date_check = r"^\s*\d{4}-\d{2}-\d{2}\s*$"

df_malformed = df.where(~F.col("doi").rlike(regex_doi))
df_null_doi = df.where(F.col("doi").isNull())

df_fine = df.select("*").where(
    F.col("doi").isNotNull() &
    F.col("doi").rlike(regex_doi)
    )
df_date_malformed = df_fine.where(
    ~F.col("date").rlike(regex_date_check)
    )
df_fine = df_fine.where(
    F.col("date").rlike(regex_date_check)
    )
df_fine = df_fine.dropDuplicates(["doi"])

In [ ]:
print(df.count())
print(df_fine.count())
print(df_fine.dropDuplicates(["doi"]).count())

4842236
1030687
1030687


In [ ]:
df_fine.columns

['title', 'doi', 'abstract', 'journal', 'date', 'authors', 'year', 'keyword']

In [44]:
# df_fine
# doi: remove ' ' from beggining of string,
# title remove [] from start-end of string, remove ' ' from begining of string, remove I. VIII. II. string beginng,
# journal: remove: ' ' from begining of st., remove (something),
# author: change ' ' to '.'
# date, year to date format
# keywords: remove [] and ''
# abstract: remove: "" from begining, and duplicated in string, remove 1. 2. 3., (1), 1)
# Final cleaning
#.withColumn("journal", regexp_replace(col("journal"), r"\(.*?\)", ""))\
df_final = df_fine\
.select("*")\
.withColumn("doi", F.trim(col("doi")))\
.withColumn("title", F.translate(col("title"), "[]", ''))\
.withColumn("title", F.regexp_replace(col("title"), r"^\s*[IVXLCDM]+\.\s*", ""))\
.withColumn("title", F.regexp_replace(col("title"), r'^"|"$', ""))\
.withColumn("title", regexp_replace(col("title"), r'""', '"'))\
.withColumn("title", F.trim(col("title")))\
.withColumn("title", F.upper(col("title")))\
.withColumn("journal", F.trim(col("journal")))\
.withColumn("abstract", regexp_replace(col("abstract"), r'^"|"$', ""))\
.withColumn("abstract", regexp_replace(col("abstract"), r'""', '"'))\
.withColumn("abstract", regexp_replace(col("abstract"), r'^\d*\.', ''))\
.withColumn("abstract", F.trim(col("abstract")))\
.withColumn("date", F.to_date(col("date"), "yyyy-MM-dd"))\
.withColumn("year", F.year(col("date")))\
.withColumn("authors", F.trim(col("authors")))\
.withColumn("authors", F.translate(col("authors"), " ", ""))\
.withColumn("authors", F.translate(col("authors"), "[]", ""))\
.withColumn("authors", F.translate(col("authors"), "'", ""))\
.withColumn("authors", F.translate(col("authors"), "", ""))\
.withColumn("authors", F.translate(col("authors"), ' ', ""))\
.withColumn("authors", F.split("authors", ","))\
.withColumn("keyword", F.translate(col("keyword"), " ", ""))\
.withColumn("keyword", F.translate(col("keyword"), "[]", ""))\
.withColumn("keyword", F.translate(col("keyword"), "'", ""))\
.withColumn("keyword", F.translate(col("keyword"), '"', ""))\
.withColumn("keyword", F.lower(col("keyword")))\
.withColumn("keyword", F.split("keyword", ","))


In [25]:
df_final.printSchema()

root
 |-- title: string (nullable = true)
 |-- doi: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- journal: string (nullable = true)
 |-- date: date (nullable = true)
 |-- authors: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- year: integer (nullable = true)
 |-- keyword: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [ ]:
# Check cleaned data on 'big' sample size
df_final.select("*").where(F.col("abstract").isNotNull()).show(1000, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Check number of partitions
df_final.rdd.getNumPartitions()

15

In [45]:
# Move journals to sep df and do mapping
uniq_journals = df_final.select("journal").distinct().na.drop()
windowSpec = Window.orderBy("journal")
df_journals = uniq_journals.withColumn("journal_id", F.row_number().over(windowSpec))

df_final = df_final.join(df_journals, on="journal", how="left").drop("journal")

In [46]:
df_final.show(20)

+--------------------+--------------------+--------------------+----------+--------------------+----+--------------------+----------+
|               title|                 doi|            abstract|      date|             authors|year|             keyword|journal_id|
+--------------------+--------------------+--------------------+----------+--------------------+----+--------------------+----------+
|HYPOPARATHYROIDIS...|10.1001/archderm....|                NULL|1971-06-01|[JPFields, LFrago...|1971|[hairloss, alopecia]|       947|
|PERINEVOID ALOPEC...|10.1001/archderm....|We report three c...|1976-10-01|[PYesudian, ASTha...|1976|[hairloss, alopecia]|       947|
|KERATOSIS FOLLICU...|10.1001/archderm....|A 10-month-old ma...|1978-05-01|[HBritton, JLusti...|1978|[urinarytractinfe...|       947|
|PSORIASIS THERAPY...|10.1001/archderm....|A study was perfo...|1983-02-01|[BPNusbaum, EKEdw...|1983|          [allergic]|       947|
|CONTINUING INCREA...|10.1001/archderm....|                NUL

In [ ]:
# Saving dataframes
df_final.write.format("csv").option("header", True).mode("overwrite").save("df_final.csv")
df_journals.write.format("csv").option("header", True).mode("overwrite").save("df_journals.csv")


In [ ]:
df_date_malformed.write.format("csv").option("header", True).mode("overwrite").save("df_date_malformed.csv")

In [ ]:
df_null_doi.write.format("csv").option("header", True).mode("overwrite").save("df_null_doi.csv")

In [ ]:
df_malformed.write.format("csv").option("header", True).mode("overwrite").save("df_malformed.csv")

In [ ]:
df_final.coalesce(1).write.format("csv").option("header", True).mode("overwrite").save("df_final.csv")

In [ ]:
#!/bin/bash
!zip archive.zip df_final.csv df_journals.csv

  adding: df_final.csv/ (stored 0%)
